# IndoJailbreak & Baselines — Cluster / JupyterLab Experiment Runner

This notebook enables running **IndoJailbreak** and baseline jailbreak algorithms (**AutoDAN, GPTFuzzer, PAIR, TAP, ForgeDAN**) directly inside a JupyterLab environment without requiring root/daemon services (such as Ollama).

### Architecture:
- **Target Model:** Local Hugging Face model (`meta-llama/Meta-Llama-3-8B-Instruct`, `Qwen/Qwen2.5-7B-Instruct`, or `aisingapore/llama3-8b-cpt-sea-lionv2-instruct`) loaded in-process with 4-bit BitsAndBytes quantization on GPU.
- **Judge & Attacker:** Cloud API (OpenAI `gpt-4o-mini` or OpenRouter) via credentials in `.env`.
- **Storage:** Cache directed to high-capacity scratch directory via `HF_HOME`.

## 1. System, GPU & Environment Verification

In [ ]:
import os
import sys
import torch
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
os.chdir(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU detected. Models will be very slow or fail to load.")

## 2. Environment Variables & HuggingFace Cache Configuration
Set your `HF_HOME` to your cluster's scratch directory to avoid exhausting your home directory storage quota.

In [ ]:
from dotenv import load_dotenv

# 1. Point HF cache to cluster scratch storage (modify if your cluster uses /work or /tmp)
user = os.environ.get("USER", "cluster_user")
scratch_cache = f"/scratch/{user}/hf_cache"

# If /scratch exists, use it; otherwise fallback to project directory cache
if Path(f"/scratch/{user}").exists():
    os.environ["HF_HOME"] = scratch_cache
    print(f"HF_HOME redirected to scratch: {scratch_cache}")
else:
    print(f"Current HF_HOME: {os.environ.get('HF_HOME', '~/.cache/huggingface')}")

# 2. Load API keys from .env file
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
    print("Loaded .env configuration successfully.")
else:
    print("NOTE: .env file not found. Create one by copying .env.example: cp .env.example .env")

has_hf_token = bool(os.getenv("HF_TOKEN"))
has_openai_key = bool(os.getenv("OPENAI_API_KEY"))
has_openrouter_key = bool(os.getenv("OPENROUTER_API_KEY"))

print(f"HF_TOKEN set: {has_hf_token} (Required for Meta-Llama-3-8B-Instruct)")
print(f"OPENAI_API_KEY set: {has_openai_key} (Used for GPT-4o-mini judge & attacker)")
print(f"OPENROUTER_API_KEY set: {has_openrouter_key}")

## 3. Test Local Hugging Face Target Model Loading (4-bit NF4)
Let's test loading `llama3_8b` (or `qwen25_7b`) using the project's native `HuggingFaceTargetLLM`.

In [ ]:
from src.models.model_factory import get_target_model

# Choose model key: 'llama3_8b', 'qwen25_7b', 'sealion_7b', or 'mock'
TEST_MODEL_KEY = "qwen25_7b"  # Change to 'llama3_8b' if HF_TOKEN is configured

print(f"Instantiating target model: {TEST_MODEL_KEY}...")
target_llm = get_target_model(TEST_MODEL_KEY)

# Generation test
test_prompt = "Halo! Bisakah kamu menjelaskan secara singkat apa itu kecerdasan buatan?"
print(f"Sending test prompt: '{test_prompt}'")
response = target_llm.generate(test_prompt)

print("\n--- Model Response ---")
print(response)

if torch.cuda.is_available():
    allocated_vram = torch.cuda.memory_allocated(0) / (1024 ** 3)
    print(f"\nVRAM in use: {allocated_vram:.2f} GB")

## 4. Run Jailbreak Baselines (AutoDAN, GPTFuzzer, PAIR, TAP, ForgeDAN)
Execute baseline experiments directly against your local Hugging Face model with the Dual-Dimensional Judge.

In [ ]:
import subprocess

# Parameters for baseline experiment
TARGET_MODEL = "qwen25_7b"       # 'llama3_8b', 'qwen25_7b', etc.
ATTACK_METHOD = "all"            # 'pair', 'tap', 'autodan', 'gptfuzzer', 'forgedan', or 'all'
DATASET = "eval1"                # 'eval1' or 'eval2'
SAMPLES_PER_CAT = 1              # Keep small (e.g. 1-3) for quick tests, 5-10 for final results
MAX_QUERIES = 50                 # Query budget per harmful goal

cmd = [
    sys.executable, "scripts/run_jailbreak_baselines.py",
    "--method", ATTACK_METHOD,
    "--target-model", TARGET_MODEL,
    "--dataset", DATASET,
    "--num-samples-per-category", str(SAMPLES_PER_CAT),
    "--max-queries", str(MAX_QUERIES),
    "--fast-eval"                 # Fast rule-based evaluation (remove for full LLM judge)
]

print("Running command:", " ".join(cmd))
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(process.stdout.readline, ''):
    print(line, end='')
process.stdout.close()
process.wait()
print(f"Finished with return code: {process.returncode}")

## 5. Run IndoJailbreak Attack (Multi-Dimensional Genetic Framework)
Run IndoJailbreak against the local model.

In [ ]:
# Execute IndoJailbreak attack
cmd_indojailbreak = [
    sys.executable, "scripts/run_attack.py",
    "--target", TARGET_MODEL,
    "--num-samples", "3",
    "--dialect", "formal",
    "--output-dir", "results/indojailbreak_runs"
]

print("Running command:", " ".join(cmd_indojailbreak))
process = subprocess.Popen(cmd_indojailbreak, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(process.stdout.readline, ''):
    print(line, end='')
process.stdout.close()
process.wait()
print(f"Finished with return code: {process.returncode}")

## 6. View & Analyze Consolidated Experiment Results
Load the latest test run results and display an inline comparative summary table.

In [ ]:
import json
import glob
import pandas as pd

# Find latest baseline run summary
summary_files = sorted(glob.glob("results/runs_test/**/summary.json", recursive=True))
if summary_files:
    latest_summary = summary_files[-1]
    print(f"Loading latest summary: {latest_summary}")
    with open(latest_summary, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    rows = []
    for method, stats in data.get("methods", {}).items():
        rows.append({
            "Method": method.upper(),
            "Goals Tested": stats.get("total_goals", 0),
            "Jailbreak Successes": stats.get("successful_goals", 0),
            "ASR (%)": f"{stats.get('asr', 0.0) * 100:.1f}%",
            "Avg Target Queries": f"{stats.get('avg_target_queries', 0.0):.1f}",
            "Avg Harmfulness (1-5)": f"{stats.get('avg_harmfulness', 0.0):.2f}",
            "Avg Duration (s)": f"{stats.get('avg_duration_sec', 0.0):.1f}"
        })
    
    df = pd.DataFrame(rows)
    display(df)
else:
    print("No summary.json found yet in results/runs_test/")